In [4]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()
pp.load_config('default_config.toml')

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc/>AGATCGGA')\
                  .named('template_pool').stylize(style='lower', which='contents')

mutated_pool = template_pool.stylize(region='cre', style='goldenrod')\
                            .mutagenize(region='cre',
                                        num_mutations=2, 
                                        style='yellow bold underline lower',
                                        mode='random', 
                                        num_states=5, 
                                        prefix='mutagenize').named('mutated_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)

L = len('GGAAAGCGGGCAGTGAGCACACAGGA') 
recomb_pool = template_pool.recombine(region='cre', 
                                      num_breakpoints=3,
                                      sources=['A'*L, 'C'*L, 'G'*L, 'T'*L],
                                      styles=['palegreen', 'springgreen', 'limegreen', 'forestgreen'],
                                      style_by='order',
                                      mode='random',
                                      num_states=5,
                                      prefix='recomb').named('recomb_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)


deletion_pool = template_pool.stylize(region='cre', style='salmon')\
                             .deletion_scan(region='cre', 
                                            deletion_length=6, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style='red bold',
                                            prefix='delscan').named('deletion_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)

sites_pool=pp.from_seqs(['AAAAAA','TTTTTT'], 
                        mode='sequential', 
                        iter_order=-1).named('sites_pool')

insertion_pool = template_pool.stylize(region='cre', style='blue')\
                              .insertion_scan(region='cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='insscan',
                                              prefix_position='pos', 
                                              prefix_insert='ins',
                                              style='cyan bold').named('insertion_pool')
                              
shuffle_pool = template_pool.stylize(region='cre', style='purple')\
                            .shuffle_scan(region='cre', 
                                          shuffle_length=6, 
                                          shuffles_per_position=2,
                                          positions=slice(None, None, 5), 
                                          mode='sequential', 
                                          style='magenta bold',
                                          prefix='shufscan',
                                          prefix_position='pos',
                                          prefix_shuffle='shuf')
                            

combo_pool = pp.stack([mutated_pool, recomb_pool, deletion_pool, insertion_pool, shuffle_pool])\
    .named('stack_pool')\
    .insert_kmers(region='bc', mode='random', length=5, prefix='bc', style='green bold')\
    .named('combo_pool')\
    .stylize(which='tags', style='gray')

pp.toggle_styles(on=True)
combo_pool.print_library(show_name=True, seed=42)

pool[27]: seq_length=None, num_states=50
state  name                           seq
    0  mutagenize_0.v_0.bc_0          tcccgact<cre>ggaaaccgggcagtgagcagacagga</cre>attacgg<bc>GGGAC</bc>agatcgga
    1  mutagenize_0.v_1.bc_1          tcccgact<cre>ggaaaccgggcagtgagcagacagga</cre>attacgg<bc>ATCGG</bc>agatcgga
    2  mutagenize_1.v_0.bc_2          tcccgact<cre>gggaagcgggcagtgaggacacagga</cre>attacgg<bc>GGCTT</bc>agatcgga
    3  mutagenize_1.v_1.bc_3          tcccgact<cre>gggaagcgggcagtgaggacacagga</cre>attacgg<bc>TGAGC</bc>agatcgga
    4  mutagenize_2.v_0.bc_4          tcccgact<cre>ggaaagcgggcagtgagtagacagga</cre>attacgg<bc>TAGAA</bc>agatcgga
    5  mutagenize_2.v_1.bc_5          tcccgact<cre>ggaaagcgggcagtgagtagacagga</cre>attacgg<bc>ACCTT</bc>agatcgga
    6  mutagenize_3.v_0.bc_6          tcccgact<cre>gaaaagcgggcagtgagcacgcagga</cre>attacgg<bc>ACGTA</bc>agatcgga
    7  mutagenize_3.v_1.bc_7          tcccgact<cre>gaaaagcgggcagtgagcacgcagga</cre>attacgg<bc>AAAAG</bc>agatcgga
    8  mutage

Pool(id=27, name='pool[27]', op='op[27]:stylize', num_states=50)

In [2]:
pp.toggle_cards(on=False)
df = combo_pool.generate_library(report_design_cards=True)
print(df.columns)
df.tail()

Index(['name', 'seq'], dtype='object')


,name,seq
45,shufscan_5.pos_2.shuf_1.bc_45,TCCCGACT<cre>GGAAAGCGGGGTGCAAGCACACAGGA</cre>A...
46,shufscan_6.pos_3.shuf_0.bc_46,TCCCGACT<cre>GGAAAGCGGGCAGTGACACAGCAGGA</cre>A...
47,shufscan_7.pos_3.shuf_1.bc_47,TCCCGACT<cre>GGAAAGCGGGCAGTGCACAAGCAGGA</cre>A...
48,shufscan_8.pos_4.shuf_0.bc_48,TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACAAGGAC</cre>A...
49,shufscan_9.pos_4.shuf_1.bc_49,TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACGGAAAC</cre>A...


In [3]:
combo_pool.print_dag()

pool[27] (pool, n=50)
└── op[27]:stylize [mode=fixed, n=1]
    └── combo_pool (pool, n=50)
        └── op[26]:get_kmers [mode=random, n=50]
            └── stack_pool (pool, n=50)
                └── op[25]:stack [mode=sequential, n=5]
                    ├── pool[4] (pool, n=10)
                    │   └── op[4]:repeat [mode=sequential, n=2]
                    │       └── mutated_pool (pool, n=5)
                    │           └── op[3]:mutagenize [mode=sequential, n=5]
                    │               └── pool[2] (pool, n=1)
                    │                   └── op[2]:stylize [mode=fixed, n=1]
                    │                       └── pool[1] (pool, n=1)
                    │                           └── op[1]:stylize [mode=fixed, n=1]
                    │                               └── template_pool (pool, n=1)
                    │                                   └── op[0]:from_seq [mode=fixed, n=1]
                    ├── pool[10] (pool, n=10)
             

Pool(id=27, name='pool[27]', op='op[27]:stylize', num_states=50)